# Dealer Gamma 01 — data gate

Pre-registration: `discussions/Dealer_Gamma/001_claude_branch_preregistration.md`. Code: `src/data_engineering/modeling/dealer_gamma.py`.

Two checks before anything else is read:

1. **Archive integrity** — summary of `tools/validate_deribit_history_trades.py` (run it first, see README).
2. **Reconstruction vs the collector** — for every 0DTE/1DTE instrument and 5-minute snapshot in the overlap: trailing-24h traded amount from the trade archive vs the collector's 24h `volume`, and |net customer position| vs `open_interest`.

**PASS** per currency: pooled volume ratio in [0.95, 1.05] and at most 5% of rows with |net| > OI·1.01 + 0.1. A FAIL stops the branch.

Environment: `DG_SYNTHETIC=1` (smoke run on synthetic data).

In [1]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
load_dotenv()
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from data_engineering.modeling import dealer_gamma as dg

ENV = dg.notebook_env('01', PROJECT_ROOT)
OUT, CACHE, HIST = ENV['out'], ENV['cache'], ENV['history']
PERIODS = ['discovery'] + (['holdout'] if ENV['holdout'] else [])
print(dg.VERSION, '| synthetic' if ENV['synthetic'] else '', '| periods', PERIODS)
print('data', ENV['data_root']); print('cache', CACHE); print('out', OUT)

DG-0.1.0  | periods ['discovery']
data D:\MarketData\Deribit
cache D:\MarketData\Deribit\derived\dealer_gamma
out D:\Projects\deribit_market_data\notebooks\Dealer_Gamma\01


## 1. Archive validator summary

In [2]:
ISSUES = []
for kind in ('option', 'future'):
    p = OUT / f'validator_{kind}' / 'deribit_daily_qc.csv'
    if not p.is_file():
        print(f'{kind}: validator output not found at {p} -> run the validator first (README step 2)'); continue
    q = pd.read_csv(p)
    print(f'{kind}:', q.groupby('currency').status.value_counts().unstack(fill_value=0).to_dict('index'))
    bad = q[q.status.isin(['FAIL', 'UNREADABLE', 'MISSING_FILE', 'IN_PROGRESS_TMP'])]
    if 'millisecond_page_limit_buckets' in q:
        bad = pd.concat([bad, q[pd.to_numeric(q.millisecond_page_limit_buckets, errors='coerce') > 0]])
    ISSUES.append(bad.assign(kind=kind))
ISSUES = pd.concat(ISSUES).drop_duplicates() if ISSUES else pd.DataFrame()
ISSUES.to_csv(OUT / '01_validator_issues.csv', index=False)
print('days with issues:', len(ISSUES)); display(ISSUES.head(30))

option: validator output not found at D:\Projects\deribit_market_data\notebooks\Dealer_Gamma\01\validator_option\deribit_daily_qc.csv -> run the validator first (README step 2)
future: validator output not found at D:\Projects\deribit_market_data\notebooks\Dealer_Gamma\01\validator_future\deribit_daily_qc.csv -> run the validator first (README step 2)
days with issues: 0


""


## 2. Reconstruction vs collector

In [3]:
GATE, BYI, CMP = {}, [], {}
for c in dg.CURRENCIES:
    coll = dg.load_collector(ENV['data_root'], c)
    print(c, 'collector rows', len(coll), 'instruments', coll.name.nunique() if len(coll) else 0)
    cmp = dg.gate_compare(HIST, coll, c)
    CMP[c] = cmp
    GATE[c] = dg.gate_summary(cmp)
    if len(cmp):
        v = cmp[cmp.vol24_coll > 0]
        BYI.append(cmp.groupby(['currency', 'name', 'grid']).agg(
            rows=('t', 'size'), oi_last=('oi', 'last'), net_last=('net_recon', 'last'),
            vol_ratio=('vol24_recon', lambda s: s.sum()), vol_coll=('vol24_coll', 'sum')).reset_index())
pd.concat(BYI).assign(vol_ratio=lambda d: d.vol_ratio / d.vol_coll.where(d.vol_coll > 0)).to_csv(
    OUT / '01_gate_by_instrument.csv', index=False) if BYI else None
display(pd.DataFrame(GATE).T)

KeyError: 'volume'

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 9))
for j, c in enumerate(dg.CURRENCIES):
    cmp = CMP[c]
    if cmp.empty:
        continue
    v = cmp[cmp.vol24_coll > 0]
    ax[0, j].hist((v.vol24_recon / v.vol24_coll).clip(0, 2), bins=80)
    ax[0, j].axvline(1, color='k', lw=1); ax[0, j].set_title(f'{c}: reconstructed / collector 24h volume')
    s = cmp.sample(min(len(cmp), 20000), random_state=0)
    ax[1, j].scatter(s.oi + 0.1, np.abs(s.net_recon) + 0.1, s=2, alpha=0.3)
    lim = [0.1, max(1, s.oi.max() * 1.2)]
    ax[1, j].plot(lim, lim, 'k-', lw=1); ax[1, j].set_xscale('log'); ax[1, j].set_yscale('log')
    ax[1, j].set_xlabel('collector open interest (coin)'); ax[1, j].set_ylabel('|net customer| reconstructed')
    ax[1, j].set_title(f'{c}: points above the line violate |net| <= OI')
plt.tight_layout(); plt.savefig(OUT / '01_gate.png', dpi=110); plt.show()

In [ ]:
STATUS = 'PASS' if all(GATE[c].get('status') == 'PASS' for c in dg.CURRENCIES) else 'FAIL'
dg.dump({'registration': dg.registration(), 'gate': GATE, 'validator_issue_days': int(len(ISSUES)),
         'status': STATUS}, OUT / '01_gate.json')
print('GATE:', STATUS)